# Supervised Analysis

This notebook develops the supervised classification workflow using the fixed development dataset and retained geochemical predictors. China remains completely withheld from model development and is reserved for final validation. The predictor set is fixed upstream and no additional feature selection is performed in this notebook.

## 1. Load Modelling Dataset

Load the log transformed modelling dataset for Random Forest classification. Z score standardisation is not required for Random Forest because tree based models are insensitive to monotonic feature scaling.

In [ ]:
# =============================================================================
# Environment setup and load development dataset
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
EXPORT_DIR = ROOT_DIR / "exports"
FIGURE_DIR = ROOT_DIR / "figures"

EXPORT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 99

DATA_FILE = PROCESSED_DIR / "Modelling_Dataset_Log.csv" #loading the log and not ZScaled
ELEMENT_FILE = EXPORT_DIR / "04_retained_trace_elements.csv"

df = pd.read_csv(DATA_FILE, encoding="utf-8")
ELEMENTS = pd.read_csv(ELEMENT_FILE, encoding="utf-8")["Element"].tolist()

META = ["Ref", "Country", "Series"]
X = df[ELEMENTS].copy()
y = df["Series"].copy()

print(f"Development analyses : {len(df)}")
print(f"Predictor elements   : {len(ELEMENTS)}")
print(ELEMENTS)
print("\nSeries counts:")
print(y.value_counts())

## 2. Verify Modelling Dataset

Check that the fixed predictor matrix contains no missing values and summarise the class and geographic composition of the development dataset before defining the supervised modelling strategy.

In [ ]:
# =============================================================================
# Verify modelling dataset
# =============================================================================

print(f"X shape           : {X.shape}")
print(f"Missing predictors: {X.isna().sum().sum()}")
print(f"Classes           : {sorted(y.unique())}")
print(f"Countries         : {df['Country'].nunique()}")

summary = pd.crosstab(df["Country"], df["Series"])
summary["Total"] = summary.sum(axis=1)
display(summary.sort_values("Total", ascending=False))

## 3. Development–Test Split

The development dataset was randomly partitioned into training (80%) and internal test (20%) subsets. Stratification by tectonic series was used to preserve the Orogenic–Anorogenic class proportions in both subsets. This sample-level split was chosen because the intended application is classification of lamproites of unknown geographic origin, rather than specifically prediction of unseen geographic groups.

The training subset is used for model fitting and subsequent hyperparameter optimisation, whereas the internal test subset remains unseen during model development and is used to evaluate predictive performance. The independently withheld Chinese dataset is not included at any stage of this procedure and is reserved for final validation.

In [ ]:
# =============================================================================
# Development–test split
# =============================================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"Training analyses : {len(X_train)}")
print(f"Test analyses     : {len(X_test)}")

split_summary = pd.DataFrame({
    "Development": y.value_counts(),
    "Training": y_train.value_counts(),
    "Test": y_test.value_counts()
})
display(split_summary)

## 4. Baseline Random Forest

A baseline Random Forest classifier was trained using the default model hyperparameters to establish a reference level of predictive performance before hyperparameter optimisation. Performance was evaluated on the untouched internal test subset using overall accuracy, balanced accuracy, class-specific precision, recall and F1 score, and ROC AUC.

In [ ]:
# =============================================================================
# Baseline Random Forest
# =============================================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

baseline_rf = RandomForestClassifier(random_state=RANDOM_STATE)
baseline_rf.fit(X_train, y_train)

y_pred = baseline_rf.predict(X_test)
y_prob = baseline_rf.predict_proba(X_test)[:, list(baseline_rf.classes_).index("Orogenic")]

baseline_metrics = pd.DataFrame([{
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced_Accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision_Orogenic": precision_score(y_test, y_pred, pos_label="Orogenic"),
    "Recall_Orogenic": recall_score(y_test, y_pred, pos_label="Orogenic"),
    "F1_Orogenic": f1_score(y_test, y_pred, pos_label="Orogenic"),
    "ROC_AUC": roc_auc_score((y_test == "Orogenic").astype(int), y_prob)
}]).round(3)

display(baseline_metrics)
print(classification_report(y_test, y_pred, digits=3))

> **Performance metric definitions**
>
> The performance of the Random Forest classifier was evaluated using several complementary metrics:
>
> - **Accuracy** measures the overall proportion of correctly classified samples. Values range from 0 to 1, where 1 indicates perfect classification.
>
> - **Balanced Accuracy** is the average recall across all classes, providing a more reliable measure than overall accuracy when class sizes are unequal. Values range from 0 to 1, with higher values indicating better balanced performance.
>
> - **Receiver Operating Characteristic Area Under the Curve (ROC AUC)** quantifies the model's ability to discriminate between classes across all possible probability thresholds. Values range from 0.5 (random classification) to 1.0 (perfect discrimination).
>
> - **Precision** measures the proportion of samples predicted to belong to a class that are correctly classified. High precision indicates a low rate of false positive predictions.
>
> - **Recall** (Sensitivity) measures the proportion of true samples belonging to a class that are correctly identified by the classifier. High recall indicates a low rate of false negative predictions.
>
> - **F1 Score** is the harmonic mean of precision and recall, providing a single measure that balances both metrics. Values range from 0 to 1, with higher values indicating better overall classification performance.

## 5. Cross-Validation

The stability of the baseline Random Forest was evaluated using ten-fold stratified cross-validation applied only to the training subset. The training data were divided into ten folds while preserving the Orogenic–Anorogenic class proportions. In each iteration, nine folds were used to fit the model and the remaining fold was used for validation, such that every training analysis was evaluated once.

Performance was assessed using accuracy, balanced accuracy and ROC AUC. The mean cross-validation performance provides an estimate of expected model behaviour across different data partitions, while the standard deviation measures sensitivity to the particular training subset. The internal test subset was not used during cross-validation.

In [ ]:
# =============================================================================
# Ten-fold stratified cross-validation
# =============================================================================
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
scoring = { "Accuracy": "accuracy", "Balanced_Accuracy": "balanced_accuracy", "ROC_AUC": "roc_auc" }
cv_results = cross_validate( baseline_rf, X_train, y_train, cv=cv, scoring=scoring )
cv_scores = pd.DataFrame({ metric: cv_results[f"test_{metric}"] for metric in scoring 
}, index=[f"Fold {i}" for i in range(1, 11)]).round(3)

cv_summary = pd.DataFrame({
    "Mean": cv_scores.mean(),
    "SD": cv_scores.std()
}).round(3)

display(cv_scores)
display(cv_summary)

> **Ten-fold stratified cross-validation**
>
> Ten-fold cross-validation was used to evaluate the stability of the baseline Random Forest within the training dataset.
>
> - The training dataset was divided into **10 approximately equal folds**.
>
> - **Stratification** preserved the relative proportions of Orogenic and Anorogenic samples in each fold.
>
> - The model was trained **10 separate times**. In each iteration, **9 folds (~90%)** were used for model fitting and the remaining **1 fold (~10%)** was used for validation.
>
> - Each training analysis therefore served as validation data **once**, while being used for model fitting in the other nine iterations.
>
> - Performance was recorded independently for each fold using **Accuracy, Balanced Accuracy and ROC AUC**.
>
> - The **mean** across the 10 folds represents the typical cross-validation performance, while the **standard deviation** indicates how sensitive model performance is to different subsets of the training data.
>
> - Cross-validation is performed **only on the 80% training subset**. The 20% internal test subset remains completely unused during cross-validation and is retained for subsequent evaluation of the developed model.
>
> - The independently withheld **Chinese dataset is not involved in either training or cross-validation** and is reserved for final validation.

## 6. Hyperparameter Optimisation

The predictive performance of a Random Forest classifier depends on a number of user-defined settings known as **hyperparameters**. Unlike model parameters, which are learned automatically during training, hyperparameters are specified before model fitting and control how the model is constructed. Appropriate optimisation of these hyperparameters can improve predictive accuracy, reduce overfitting and increase model robustness.

Hyperparameter optimisation was performed using **GridSearchCV**, which systematically evaluates every combination of predefined hyperparameter values. Each candidate model was assessed using **ten-fold stratified cross-validation**, ensuring that the class proportions were preserved within each fold. The optimal hyperparameter combination was selected based on the highest mean cross-validation performance.

The principal Random Forest hyperparameters investigated in this study are:

- **`n_estimators`** – Number of decision trees constructed within the forest. Increasing the number of trees generally improves model stability, although computational cost also increases.

- **`max_depth`** – Maximum depth of each decision tree. Deeper trees can capture more complex relationships but may increase the risk of overfitting.

- **`min_samples_split`** – Minimum number of samples required before a node can be divided. Larger values produce simpler trees by preventing unnecessary splits.

- **`min_samples_leaf`** – Minimum number of samples permitted within a terminal leaf node. Increasing this value reduces model complexity and can improve generalisation.

- **`max_features`** – Number of predictor variables randomly considered at each node split. This controls the diversity of individual trees within the Random Forest ensemble.

Each combination of hyperparameter values was evaluated jointly using GridSearchCV, and the combination yielding the highest mean cross-validated balanced accuracy was selected for the optimised Random Forest classifier.

In [ ]:
# =============================================================================
# Hyperparameter optimisation
# =============================================================================
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.ensemble import RandomForestClassifier
from tqdm.auto import tqdm
from contextlib import contextmanager
import joblib

BEST_RF_PATH = EXPORT_DIR / "best_rf.joblib"

@contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()

if BEST_RF_PATH.exists():
    best_rf = joblib.load(BEST_RF_PATH)

    print(f"Loaded existing optimised Random Forest: {BEST_RF_PATH}")
    print("\nBest hyperparameters:")
    for parameter in [
        "max_depth",
        "max_features",
        "min_samples_leaf",
        "min_samples_split",
        "n_estimators"
    ]:
        print(f"{parameter}: {best_rf.get_params()[parameter]}")

    print("\nHyperparameter optimisation skipped.")

else:
    param_grid = {
        "n_estimators": [300, 500, 800],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.5]
    }

    n_combinations = len(list(ParameterGrid(param_grid)))
    n_fits = n_combinations * cv.get_n_splits()

    print(f"Hyperparameter combinations : {n_combinations}")
    print(f"Cross-validation folds      : {cv.get_n_splits()}")
    print(f"Total model fits            : {n_fits}")

    grid_search = GridSearchCV(
        estimator=RandomForestClassifier(random_state=RANDOM_STATE),
        param_grid=param_grid,
        scoring="balanced_accuracy",
        cv=cv,
        n_jobs=-1,
        verbose=0
    )

    with tqdm_joblib(tqdm(total=n_fits, desc="Grid search", unit="fit")):
        grid_search.fit(X_train, y_train)

    best_rf = grid_search.best_estimator_

    print("\nBest hyperparameters:")
    for parameter, value in grid_search.best_params_.items():
        print(f"{parameter}: {value}")

    print(f"\nBest CV balanced accuracy: {grid_search.best_score_:.3f}")

    joblib.dump(best_rf, BEST_RF_PATH)
    print(f"\nSaved optimised Random Forest: {BEST_RF_PATH}")

## 6. Final Optimised Random Forest

The optimised Random Forest selected by GridSearchCV was evaluated on the untouched 20% internal test subset. This test dataset was not used during model fitting, cross-validation or hyperparameter optimisation and therefore provides an independent assessment of model generalisation within the development dataset.

Performance was assessed using accuracy, balanced accuracy and ROC AUC, together with class-specific precision, recall and F1 score. A confusion matrix was also generated to show the number and direction of classification errors. The optimised model was compared directly with the baseline Random Forest using the same internal test subset.

In [ ]:
# =============================================================================
# Final optimised Random Forest evaluation
# =============================================================================

from sklearn.metrics import confusion_matrix, classification_report

y_pred_opt = best_rf.predict(X_test)
y_prob_opt = best_rf.predict_proba(X_test)[:, list(best_rf.classes_).index("Orogenic")]

optimised_metrics = pd.DataFrame([{
    "Accuracy": accuracy_score(y_test, y_pred_opt),
    "Balanced_Accuracy": balanced_accuracy_score(y_test, y_pred_opt),
    "ROC_AUC": roc_auc_score((y_test == "Orogenic").astype(int), y_prob_opt)
}]).round(3)

class_metrics_opt = (
    pd.DataFrame(classification_report(y_test, y_pred_opt, output_dict=True))
    .T.loc[["Anorogenic", "Orogenic"], ["precision", "recall", "f1-score", "support"]]
    .rename(columns={
        "precision": "Precision",
        "recall": "Recall",
        "f1-score": "F1",
        "support": "N"
    })
)
class_metrics_opt[["Precision", "Recall", "F1"]] = class_metrics_opt[["Precision", "Recall", "F1"]].round(3)
class_metrics_opt["N"] = class_metrics_opt["N"].astype(int)

comparison = pd.DataFrame({
    "Baseline": [
        accuracy_score(y_test, y_pred),
        balanced_accuracy_score(y_test, y_pred),
        roc_auc_score((y_test == "Orogenic").astype(int), y_prob)
    ],
    "Optimised": [
        accuracy_score(y_test, y_pred_opt),
        balanced_accuracy_score(y_test, y_pred_opt),
        roc_auc_score((y_test == "Orogenic").astype(int), y_prob_opt)
    ]
}, index=["Accuracy", "Balanced Accuracy", "ROC AUC"]).round(3)

cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred_opt, labels=["Anorogenic", "Orogenic"]),
    index=["True Anorogenic", "True Orogenic"],
    columns=["Predicted Anorogenic", "Predicted Orogenic"]
)

display(optimised_metrics)
display(class_metrics_opt)
display(comparison)
display(cm)

## 7. Random Forest Decision Structure

Overall predictive performance does not show how individual geochemical variables are used within the Random Forest. The internal decision structure of the optimised model was therefore examined to identify which elements are repeatedly used to discriminate between Orogenic and Anorogenic lamproites.

For each element, the analysis records how often it occurs as the root node of a tree, the proportion of trees in which it is used at least once, the total number of decision splits involving that element, and the mean depth at which those splits occur. Lower mean split depths indicate that an element tends to contribute earlier in the decision process.

Because the Random Forest was trained using log10-transformed concentrations, split thresholds were converted back to the original concentration scale for interpretation. The median threshold across all splits was calculated for each element, together with the tectonic class most commonly associated with concentrations below that threshold.

These statistics describe how consistently and where within the forest each geochemical variable contributes to classification, rather than providing a single global measure of feature importance.

In [ ]:
# =============================================================================
# Random Forest decision structure
# =============================================================================

from collections import Counter

n_trees = len(best_rf.estimators_)
root_nodes, split_nodes, split_depths, low_classes = [], [], [], []
split_thresholds = {e: [] for e in ELEMENTS}

for tree in best_rf.estimators_:
    t = tree.tree_

    # Root node
    root_feature = t.feature[0]
    root_nodes.append(ELEMENTS[root_feature])

    # All decision nodes
    stack = [(0, 0)]

    while stack:
        node, depth = stack.pop()

        if t.feature[node] >= 0:
            feature = t.feature[node]
            element = ELEMENTS[feature]
            threshold = t.threshold[node]

            split_nodes.append(element)
            split_depths.append((element, depth))
            split_thresholds[element].append(10 ** threshold)

            left = t.children_left[node]
            right = t.children_right[node]

            low_class = best_rf.classes_[np.argmax(t.value[left][0])]
            low_classes.append((element, low_class))

            stack.append((left, depth + 1))
            stack.append((right, depth + 1))

root_counts = pd.Series(Counter(root_nodes))
split_counts = pd.Series(Counter(split_nodes))

tree_use = pd.Series({
    element: sum(
        element in [ELEMENTS[f] for f in tree.tree_.feature[tree.tree_.feature >= 0]]
        for tree in best_rf.estimators_
    )
    for element in ELEMENTS
})

mean_depth = (
    pd.DataFrame(split_depths, columns=["Element", "Depth"])
    .groupby("Element")["Depth"]
    .mean()
)

dominant_low_class = (
    pd.DataFrame(low_classes, columns=["Element", "Class"])
    .groupby("Element")["Class"]
    .agg(lambda x: x.value_counts().idxmax())
)

forest_stats = pd.DataFrame(index=ELEMENTS)
forest_stats["Root_Nodes"] = root_counts
forest_stats["Root_Percent"] = 100 * root_counts / n_trees
forest_stats["Trees_Using_Variable"] = tree_use
forest_stats["Trees_Using_Variable_Percent"] = 100 * tree_use / n_trees
forest_stats["Total_Splits"] = split_counts
forest_stats["Mean_Splits_per_Tree"] = split_counts / n_trees
forest_stats["Mean_Split_Depth"] = mean_depth
forest_stats["Median_Split_Threshold"] = [
    np.median(split_thresholds[e]) if split_thresholds[e] else np.nan
    for e in ELEMENTS
]
forest_stats["Dominant_Low_Value_Class"] = dominant_low_class

forest_stats[["Root_Nodes", "Trees_Using_Variable", "Total_Splits"]] = (
    forest_stats[["Root_Nodes", "Trees_Using_Variable", "Total_Splits"]]
    .fillna(0)
    .astype(int)
)

numeric_cols = [
    "Root_Percent",
    "Trees_Using_Variable_Percent",
    "Mean_Splits_per_Tree",
    "Mean_Split_Depth",
    "Median_Split_Threshold"
]
forest_stats[numeric_cols] = forest_stats[numeric_cols].round(3)

forest_stats = forest_stats.sort_values(
    ["Root_Nodes", "Total_Splits"],
    ascending=False
)

display(forest_stats)

>Root Nodes: Number of trees where this element is the very first decision.<br>
>Root (%): Percentage of the forest beginning with this element.<br>
>Trees Using Variable: Number of trees in which the element appears at least once.<br>
>Trees Using Variable (%): How consistently the forest relies on this element.<br>
>Total Splits: Total number of decision nodes using this element across all trees.<br>
>Mean Splits per Tree: On average, how often the variable is revisited within a tree.<br>
>Mean Split Depth: Average depth at which the variable is encountered. Lower = earlier decision.<br>
>Median Root Threshold: Typical concentration used when the variable is the first discriminator.<br>
>Median Split Threshold: Typical concentration whenever the variable is used.<br>
Dominant Low-Value Class: Which class most often results when the value is below the threshold.

In [ ]:
# =============================================================================
# Representative Random Forest trees in real concentration units
# =============================================================================

import copy
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# Identify representative trees
# -----------------------------------------------------------------------------

tree_stats = pd.DataFrame([
    {
        "Tree": i,
        "Root Feature": ELEMENTS[t.tree_.feature[0]],
        "Depth": t.tree_.max_depth
    }
    for i, t in enumerate(best_rf.estimators_)
])

representative_trees = pd.concat([
    tree_stats.query("`Root Feature` == @e")
    .sort_values("Depth")
    .head(1)
    for e in ["Ti", "Nb", "Ba", "La", "Zr", "Ce"]
])

# -----------------------------------------------------------------------------
# Plot trees with thresholds converted from log10 to real concentrations
# -----------------------------------------------------------------------------

for _, r in representative_trees.iterrows():

    tree_number = int(r["Tree"])
    tree_plot = copy.deepcopy(best_rf.estimators_[tree_number])

    # Convert split thresholds from log10 concentration to original units.
    # Leaf nodes have feature == -2 and are left unchanged.
    split_nodes = tree_plot.tree_.feature >= 0
    tree_plot.tree_.threshold[split_nodes] = (
        10 ** tree_plot.tree_.threshold[split_nodes]
    )

    fig, ax = plt.subplots(figsize=(80, 22))

    plot_tree(
        tree_plot,
        feature_names=ELEMENTS,
        class_names=best_rf.classes_,
        filled=True,
        rounded=True,
        impurity=False,
        proportion=False,
        precision=2,
        fontsize=32,
        ax=ax
    )

    ax.set_title(
        f"Representative Tree {tree_number} "
        f"(Root = {r['Root Feature']})",
        fontsize=24
    )

    plt.tight_layout()

    outpath = (
        FIGURE_DIR
        / f"11_RF_Representative_Tree_{tree_number:03d}_{r['Root Feature']}.pdf"
    )

    plt.savefig(outpath, bbox_inches="tight")
    plt.show()

    print(f"Saved: {outpath}")

## 8. External Validation Using the Withheld China Dataset

Following hyperparameter optimisation and evaluation on the internal test set, the selected Random Forest configuration was fixed and refitted using the complete development dataset. No further model selection or hyperparameter tuning was performed at this stage.

The final model was then evaluated using the China dataset, which had been withheld from all preceding stages of data preprocessing, feature selection, model training, cross-validation and hyperparameter optimisation. The same 14 geochemical predictors and log10 transformation used for model development were applied to the validation data.

External validation performance was assessed using accuracy, balanced accuracy, ROC AUC, class-specific precision, recall and F1 score, together with the confusion matrix and individual sample predictions.

In [ ]:
# =============================================================================
# External validation using withheld China dataset
# =============================================================================
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, classification_report, confusion_matrix
import joblib
VALIDATION_COUNTRY = "China"
RAW_DIR = DATA_DIR / "raw"

# -----------------------------------------------------------------------------
# Refit final model on all development data using fixed optimised parameters
# -----------------------------------------------------------------------------
final_rf = clone(best_rf)
final_rf.fit(X, y)
joblib.dump(final_rf, EXPORT_DIR / "final_rf.joblib")
print("Final Random Forest refitted on complete development dataset.")
print(f"Development analyses : {len(X)}")
print(f"Predictor elements   : {len(ELEMENTS)}")

# -----------------------------------------------------------------------------
# Load completely withheld China dataset
# -----------------------------------------------------------------------------
validation_df = pd.read_csv(RAW_DIR / "Data_Input.csv", encoding="utf-8")
validation_df["Ti"] = validation_df["TiO2"] * 0.5994 * 1e4
validation_df["K"] = validation_df["K2O"] * 0.8301 * 1e4
validation_df["P"] = validation_df["P2O5"] * 0.4364 * 1e4
validation_df = validation_df[validation_df["Country"].eq(VALIDATION_COUNTRY)].copy()

missing_elements = [e for e in ELEMENTS if e not in validation_df.columns]
if missing_elements:
    raise ValueError(f"Missing validation predictors: {missing_elements}")
if validation_df[ELEMENTS].isna().any().any():
    raise ValueError("Missing predictor values are present in the China validation dataset.")
if (validation_df[ELEMENTS] <= 0).any().any():
    raise ValueError("Non-positive predictor values are present in the China validation dataset.")

X_validation = np.log10(validation_df[ELEMENTS].copy())
y_validation = validation_df["Series"].copy()

print(f"\nValidation analyses  : {len(validation_df)}")
print("Validation class counts:")
print(y_validation.value_counts())

y_pred_validation = final_rf.predict(X_validation)
orogenic_idx = list(final_rf.classes_).index("Orogenic")
y_prob_validation = final_rf.predict_proba(X_validation)[:, orogenic_idx]

validation_metrics = pd.DataFrame([{
    "Accuracy": accuracy_score(y_validation, y_pred_validation),
    "Balanced_Accuracy": balanced_accuracy_score(y_validation, y_pred_validation),
    "ROC_AUC": roc_auc_score((y_validation == "Orogenic").astype(int), y_prob_validation)
}]).round(3)

validation_class_metrics = (
    pd.DataFrame(classification_report(y_validation, y_pred_validation, output_dict=True))
    .T.loc[["Anorogenic", "Orogenic"], ["precision", "recall", "f1-score", "support"]]
    .rename(columns={"precision": "Precision", "recall": "Recall", "f1-score": "F1", "support": "N"})
)
validation_class_metrics[["Precision", "Recall", "F1"]] = validation_class_metrics[["Precision", "Recall", "F1"]].round(3)
validation_class_metrics["N"] = validation_class_metrics["N"].astype(int)

validation_cm = pd.DataFrame(
    confusion_matrix(y_validation, y_pred_validation, labels=["Anorogenic", "Orogenic"]),
    index=["True Anorogenic", "True Orogenic"],
    columns=["Predicted Anorogenic", "Predicted Orogenic"]
)

metadata_cols = [c for c in ["Ref", "Country", "Locality", "Series"] if c in validation_df.columns]
validation_predictions = validation_df[metadata_cols].copy()
validation_predictions["Predicted"] = y_pred_validation
validation_predictions["Probability_Orogenic"] = y_prob_validation.round(3)
validation_predictions["Correct"] = validation_predictions["Series"].eq(validation_predictions["Predicted"])

display(validation_metrics)
display(validation_class_metrics)
display(validation_cm)
display(validation_predictions)

# -----------------------------------------------------------------------------
# Export validation results
# -----------------------------------------------------------------------------

metrics_outpath = EXPORT_DIR / "16_RF_China_Validation_Metrics.csv"
predictions_outpath = EXPORT_DIR / "17_RF_China_Validation_Predictions.csv"
cm_outpath = EXPORT_DIR / "18_RF_China_Validation_Confusion_Matrix.csv"

validation_metrics.to_csv(metrics_outpath, index=False)
validation_predictions.to_csv(predictions_outpath, index=False)
validation_cm.to_csv(cm_outpath)

print(f"\nSaved: {EXPORT_DIR / 'final_rf.joblib'}")
print(f"Saved: {metrics_outpath}")
print(f"Saved: {predictions_outpath}")
print(f"Saved: {cm_outpath}")